The goal of this notebook is to show how the interior of every tracked cyclone and anticyclone changes over the calendar record, without reference to eddy age: Copernicus CHL, the nine Copernicus plankton groups, and the 13 SDP pigments.

- Every tracked eddy of each polarity counts, not only the target eddies of the lifetime notebooks.
- Each line is the median over the eddies with a composite on that date, and the band is the interquartile range. The medians are used because a few shelf eddies carry values ten times the open-ocean ones and pull a mean.
- A group keeps the same coverage rule as `pfts_over_lifetime.ipynb` on its own pixels. The pigment record starts with PACE in March 2024.

In [ ]:
from pathlib import Path
from typing import cast
import sys

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.axes import Axes
from matplotlib.figure import Figure

PROJECT_ROOT = Path('/Users/jerry/school/research/eddy-tracking')
sys.path.insert(0, str(PROJECT_ROOT))
from eddy_tracking.config import load_config

EXPERIMENT = 'gulf_stream_20240305_20260531'
DATA_DIR = PROJECT_ROOT / 'data' / EXPERIMENT
cfg = load_config(EXPERIMENT)
polarity_colors = {'cyclone': '#2166ac', 'anticyclone': '#b2182b'}
pigments = ['Tchla', 'Zea', 'DV_chla', 'ButFuco', 'HexFuco', 'Allo', 'MV_chlb', 'Neo', 'Viola', 'Fuco', 'Chlc12', 'Chlc3', 'Perid']
pigment_labels = {
    'Tchla': 'Total chlorophyll-a', 'Zea': 'Zeaxanthin', 'DV_chla': 'Divinyl chlorophyll-a',
    'ButFuco': "19'-But-fucoxanthin", 'HexFuco': "19'-Hex-fucoxanthin", 'Allo': 'Alloxanthin',
    'MV_chlb': 'Monovinyl chlorophyll-b', 'Neo': 'Neoxanthin', 'Viola': 'Violaxanthin', 'Fuco': 'Fucoxanthin',
    'Chlc12': 'Chlorophyll-c1+c2', 'Chlc3': 'Chlorophyll-c3', 'Perid': 'Peridinin',
}
groups = ['DIATO', 'DINO', 'GREEN', 'HAPTO', 'PROCHLO', 'PROKAR', 'MICRO', 'NANO', 'PICO']
group_labels = {
    'DIATO': 'Diatoms', 'DINO': 'Dinophytes', 'GREEN': 'Green algae', 'HAPTO': 'Haptophytes',
    'PROCHLO': 'Prochlorophytes', 'PROKAR': 'Prokaryotes',
    'MICRO': 'Microphytoplankton', 'NANO': 'Nanophytoplankton', 'PICO': 'Picophytoplankton',
}
panel_letters = 'abcdefghijklm'
log_ticks = np.concatenate([np.array([1, 2, 5]) * 10.0 ** power for power in range(-5, 2)])

plankton_table = pd.read_parquet(DATA_DIR / 'gold/eddy_plankton_table.parquet')
plankton_settings = cfg['collocate_plankton']
for group in groups:
    covered = (
        plankton_table[f'{group}_n_pixels'].ge(plankton_settings['min_pixels'])
        & plankton_table[f'{group}_n_pixels'].div(plankton_table['n_pixels']).ge(plankton_settings['min_coverage'])
    )
    plankton_table.loc[~covered, group] = np.nan
pigment_table = pd.read_parquet(DATA_DIR / 'gold/eddy_pigment_table.parquet')
pigment_table['polarity'] = cast(pd.Series, pigment_table['polarity']).map({0: 'anticyclone', 1: 'cyclone'})
pigment_table = pigment_table.rename(columns={f'eddy_mean_{pigment}': pigment for pigment in pigments})

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold',
    'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
    'figure.dpi': 150, 'savefig.dpi': 300,
})


def draw_series(ax: Axes, table: pd.DataFrame, column: str) -> None:
    quartiles = table.groupby(['polarity', 'date'])[column].quantile(np.array([0.25, 0.5, 0.75])).unstack()
    for polarity, color in polarity_colors.items():
        series = quartiles.loc[polarity]
        ax.fill_between(series.index, series[0.25], series[0.75], color=color, alpha=0.15, linewidth=0)
        ax.plot(series.index, series[0.5], color=color, linewidth=1.0, label=f'{polarity.capitalize()}s ({table.loc[table["polarity"].eq(polarity), "track_id"].nunique()} tracks)')
    dates = quartiles.index.get_level_values('date')
    ax.set_xlim(dates.min(), dates.max())
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 7)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    low = log_ticks[log_ticks <= quartiles[0.25].min()].max()
    high = log_ticks[log_ticks >= quartiles[0.75].max()].min()
    ticks = log_ticks[(log_ticks >= low) & (log_ticks <= high)]
    if len(ticks) > 7:
        low, high = 10.0 ** np.floor(np.log10(low)), 10.0 ** np.ceil(np.log10(high))
        ticks = 10.0 ** np.arange(np.log10(low), np.log10(high) + 1)
    ax.set(yscale='log', ylim=(low, high))
    ax.yaxis.set_ticks(ticks, [f'{tick:g}' for tick in ticks])
    ax.yaxis.set_ticks([], minor=True)
    ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
    ax.set_axisbelow(True)

In [ ]:
fig, ax = cast(tuple[Figure, Axes], plt.subplots(figsize=(6.69, 2.6), layout='constrained'))
draw_series(ax, plankton_table, 'CHL')
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 4, 7, 10)))
ax.set_ylabel('Interior CHL (mg m$^{-3}$)')
ax.set_xlabel('Composite date')
ax.legend(loc='upper right')
plt.show()

In [ ]:
pigment_fig, pigment_axes = cast(tuple[Figure, np.ndarray], plt.subplots(7, 2, figsize=(6.69, 8.9), sharex=True, layout='constrained'))
for letter, ax, pigment in zip(panel_letters, pigment_axes.flat, pigments):
    ax = cast(Axes, ax)
    draw_series(ax, pigment_table, pigment)
    ax.set_title(f'$\\bf{{({letter})}}$ {pigment_labels[pigment]}', loc='left')
cast(Axes, pigment_axes[-2, 1]).tick_params(labelbottom=True)
legend_ax = cast(Axes, pigment_axes.flat[len(pigments)])
legend_ax.axis('off')
legend_ax.legend(*cast(Axes, pigment_axes.flat[0]).get_legend_handles_labels(), loc='upper left').set_in_layout(False)
pigment_fig.supxlabel('Composite date', fontsize=8)
pigment_fig.supylabel('Pigment concentration (mg m$^{-3}$)', fontsize=8)
plt.show()

In [ ]:
group_fig, group_axes = cast(tuple[Figure, np.ndarray], plt.subplots(5, 2, figsize=(6.69, 6.6), sharex=True, layout='constrained'))
for letter, ax, group in zip(panel_letters, group_axes.flat, groups):
    ax = cast(Axes, ax)
    draw_series(ax, plankton_table, group)
    ax.set_title(f'$\\bf{{({letter})}}$ {group_labels[group]}', loc='left')
cast(Axes, group_axes[-2, 1]).tick_params(labelbottom=True)
legend_ax = cast(Axes, group_axes.flat[len(groups)])
legend_ax.axis('off')
legend_ax.legend(*cast(Axes, group_axes.flat[0]).get_legend_handles_labels(), loc='upper left').set_in_layout(False)
group_fig.supxlabel('Composite date', fontsize=8)
group_fig.supylabel('Group chlorophyll-a (mg m$^{-3}$)', fontsize=8)
plt.show()

In [ ]:
from IPython.display import display

percentiles = np.array([1, 5, 10, 25, 50, 75, 90, 95, 99])
chl_quantiles = plankton_table.groupby('polarity')['CHL'].quantile(percentiles / 100).unstack()
chl_quantiles.columns = pd.Index(percentiles, name='percentile')

box_fig, box_ax = cast(tuple[Figure, Axes], plt.subplots(figsize=(3.35, 3.0), layout='constrained'))
for position, (polarity, color) in enumerate(polarity_colors.items()):
    values = plankton_table.loc[plankton_table['polarity'].eq(polarity), 'CHL']
    box_ax.boxplot(
        values, positions=[position], widths=0.5, whis=(5, 95),
        boxprops={'color': color, 'linewidth': 1.0}, whiskerprops={'color': color, 'linewidth': 0.8}, capprops={'color': color, 'linewidth': 0.8},
        medianprops={'color': color, 'linewidth': 1.4}, flierprops={'marker': 'o', 'markersize': 2, 'markerfacecolor': color, 'markeredgecolor': 'none', 'alpha': 0.3},
    )
box_ax.set(yscale='log', ylim=(0.02, 5))
box_ax.yaxis.set_ticks([0.02, 0.05, 0.1, 0.2, 0.5, 1, 2, 5], ['0.02', '0.05', '0.1', '0.2', '0.5', '1', '2', '5'])
box_ax.yaxis.set_ticks([], minor=True)
box_ax.xaxis.set_ticks([0, 1], [f'{polarity.capitalize()}s\n({plankton_table["polarity"].eq(polarity).sum()} eddy-composites)' for polarity in polarity_colors])
box_ax.tick_params(axis='x', length=0)
box_ax.set_ylabel('Interior CHL (mg m$^{-3}$)')
box_ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
box_ax.set_axisbelow(True)
plt.show()
display(chl_quantiles.round(3))